# Getting the Silver Data and Pushing to Gold

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
df = spark.read.table("ecom.silver.fact_silver_orderitems")
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
table_name = "ecom.gold.fact_gold_orderitems"
silver_df = df.select(
    F.col("order_id"),
    F.col("order_item_id"),
    F.col("product_id"),
    F.col("seller_id"),
    F.col("shipping_limit_date"),
    F.col("price"),
    F.col("freight_value"),
    F.col("ingest_date"),
    F.lit("dq_status"),
)
 

In [0]:

if spark.catalog.tableExists(table_name):

    gold_delta = DeltaTable.forName(spark, table_name)
    (
    gold_delta.alias("target")
    .merge(
        silver_df.alias("source"),
        """
        target.order_id = source.order_id
        AND target.order_item_id = source.order_item_id
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
    print("Rows Appended")

else:
    
    gold_df = df.select(
        F.col("order_id"),
        F.col("order_item_id"),
        F.col("product_id"),
        F.col("seller_id"),
        F.col("shipping_limit_date"),
        F.col("price"),
        F.col("freight_value"),
        F.col("ingest_date"),
        F.lit("dq_status"),
    )
    (
        gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tablename)
    )